# Profiling for LLM Infrastructure Interviews
## PyTorch Profiler, Framework-Level Tracing, Nsight Systems, and Nsight Compute

This notebook is a teaching-first, interview-focused walkthrough of the main profiling tools and ideas that matter for LLM infrastructure roles.

It is designed for:
- LLM infra interviews
- inference / training systems interviews
- performance engineering interviews
- candidates who want both intuition and working examples

## What this notebook covers

By the end, you will have:
1. a realistic tiny Transformer-like workload to profile
2. a baseline train step and inference step
3. a useful PyTorch profiler workflow
4. scheduled profiling and trace export
5. a clean mental model of framework-level tracing
6. a practical distinction between Nsight Systems and Nsight Compute
7. several small exercises with hints and answers

## Why profiling matters for infra roles

A strong infra engineer usually needs to answer questions like:
- why is the step slow?
- why is the GPU underutilized?
- is the bottleneck CPU-side or GPU-side?
- which operator dominates?
- are kernels compute-bound or memory-bound?
- is the problem in framework code, scheduling, data movement, or one hot kernel?

A good rule of thumb is:
- **PyTorch profiler**: start here for framework/operator hotspots
- **Nsight Systems**: use for whole-program CPU/GPU timeline analysis
- **Nsight Compute**: use for deep per-kernel analysis

## Official-tool mental model

### PyTorch profiler
Best for:
- operator-level hotspot ranking
- CPU vs CUDA time inside the framework
- memory and shape-aware profiling
- trace export into TensorBoard / Chrome trace

### Nsight Systems
Best for:
- system-level timeline analysis
- CPU threads, CUDA API calls, GPU kernels, copies, NVTX ranges
- identifying idle gaps and overlap problems
- answering *what happened when?*

### Nsight Compute
Best for:
- one kernel at a time
- kernel metrics such as occupancy, memory throughput, and launch details
- answering *why is this specific kernel slow?*

In [ ]:

# ============================================================
# Imports and environment checks
# ============================================================
# This notebook focuses on profiling concepts for LLM infra roles.
# We keep the workload small enough to run in Colab and on CPU-only
# machines, but realistic enough that the profiler still shows:
# - embeddings
# - attention
# - layer norm
# - MLP/GELU
#
# The main ideas we want to teach are:
# - how to profile at the framework/operator level
# - how to export traces
# - when to use PyTorch profiler
# - when to use Nsight Systems
# - when to use Nsight Compute

import math
import os
import random
import shutil
import tempfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.profiler import ProfilerActivity, profile, record_function

SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")
print(f"nsys available: {shutil.which('nsys') is not None}")
print(f"ncu available:  {shutil.which('ncu') is not None}")


## 1. Tiny workload to profile

We use a small decoder-style model with:
- embeddings
- causal self-attention
- layer norms
- MLP/GELU

That makes the profile more realistic than a plain MLP while still being small enough for Colab or CPU runs.

In [ ]:

# ============================================================
# Small Transformer-like model for profiling
# ============================================================
# We use a very small decoder-style model so the notebook stays
# runnable in Colab and on CPU.
#
# This is not a production LLM.
# It is a teaching workload designed to produce meaningful traces.

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
DIGIT_OFFSET = 3
VOCAB_SIZE = 13

def make_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
    """Return a lower-triangular causal mask."""
    return torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device))

def make_counting_batch(
    batch_size: int,
    min_digits: int = 6,
    max_digits: int = 10,
    device: torch.device = torch.device("cpu"),
) -> tuple[torch.Tensor, torch.Tensor]:
    """Build a tiny next-token language-model batch."""
    rows = []
    max_len = 0

    for _ in range(batch_size):
        start_digit = random.randint(0, 9)
        seq_len = random.randint(min_digits, max_digits)

        digits = [DIGIT_OFFSET + ((start_digit + i) % 10) for i in range(seq_len)]
        full_sequence = [BOS_ID] + digits + [EOS_ID]

        rows.append(full_sequence)
        max_len = max(max_len, len(full_sequence))

    padded_rows = [
        row + [PAD_ID] * (max_len - len(row))
        for row in rows
    ]

    full = torch.tensor(padded_rows, dtype=torch.long, device=device)
    return full[:, :-1], full[:, 1:]

class TinyAttention(nn.Module):
    def __init__(self, d_model: int = 32, num_heads: int = 4, dropout: float = 0.0):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout_p = dropout

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape
        x = x.view(batch_size, seq_len, self.num_heads, self.head_dim)
        return x.transpose(1, 2)

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, _, seq_len, _ = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(batch_size, seq_len, self.d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        with record_function("attention_qkv_projection"):
            q = self._split_heads(self.q_proj(x))
            k = self._split_heads(self.k_proj(x))
            v = self._split_heads(self.v_proj(x))

        with record_function("attention_sdpa"):
            mask = make_causal_mask(x.size(1), x.device)
            y = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask=mask,
                dropout_p=self.dropout_p if self.training else 0.0,
                is_causal=False,
            )

        with record_function("attention_output_projection"):
            y = self._merge_heads(y)
            y = self.out_proj(y)

        return y

class TinyBlock(nn.Module):
    def __init__(self, d_model: int = 32, num_heads: int = 4, ff_dim: int = 64):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.attn = TinyAttention(d_model=d_model, num_heads=num_heads, dropout=0.0)

        self.ln2 = nn.LayerNorm(d_model)
        self.fc1 = nn.Linear(d_model, ff_dim)
        self.fc2 = nn.Linear(ff_dim, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        with record_function("decoder_block_attention"):
            x = x + self.attn(self.ln1(x))

        with record_function("decoder_block_mlp"):
            y = self.fc1(self.ln2(x))
            y = F.gelu(y)
            y = self.fc2(y)
            x = x + y

        return x

class TinyCausalLM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 32, num_heads: int = 4, ff_dim: int = 64, num_layers: int = 1, max_len: int = 64):
        super().__init__()

        self.d_model = d_model
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)

        self.layers = nn.ModuleList([
            TinyBlock(d_model=d_model, num_heads=num_heads, ff_dim=ff_dim)
            for _ in range(num_layers)
        ])

        self.final_ln = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embed.weight

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = input_ids.shape

        with record_function("embedding_lookup"):
            positions = torch.arange(seq_len, device=input_ids.device)
            x = self.token_embed(input_ids) * math.sqrt(self.d_model)
            x = x + self.pos_embed(positions)[None, :, :]

        for layer in self.layers:
            x = layer(x)

        with record_function("lm_head_projection"):
            x = self.final_ln(x)
            logits = self.lm_head(x)

        return logits

model = TinyCausalLM(vocab_size=VOCAB_SIZE, d_model=32, num_heads=4, ff_dim=64, num_layers=1, max_len=64).to(device)

input_ids, target_ids = make_counting_batch(batch_size=4, device=device)
logits = model(input_ids)

print("Input shape:", tuple(input_ids.shape))
print("Logits shape:", tuple(logits.shape))


## 2. Baseline timing before profiling

Before using any profiler, establish a baseline:
- does the code work?
- how long is a step roughly?
- are you profiling training or inference?

That is a very senior-engineer habit.

In [ ]:

# ============================================================
# Baseline train and inference steps
# ============================================================
# Before profiling, establish a simple baseline.
# This is a senior-engineer habit:
# - prove the workload works
# - get a rough wall-clock
# - then use a profiler

loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

def train_step(model: nn.Module, optimizer: torch.optim.Optimizer) -> float:
    """Run one tiny training step and return the scalar loss."""
    model.train()

    with record_function("train_step_data"):
        input_ids, target_ids = make_counting_batch(batch_size=4, device=device)

    with record_function("train_step_forward"):
        logits = model(input_ids)
        loss = loss_fn(logits.reshape(-1, VOCAB_SIZE), target_ids.reshape(-1))

    with record_function("train_step_backward"):
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return float(loss.item())

@torch.inference_mode()
def inference_step(model: nn.Module) -> torch.Tensor:
    """Run one tiny inference pass and return logits."""
    model.eval()

    with record_function("inference_step_data"):
        input_ids, _ = make_counting_batch(batch_size=2, device=device)

    with record_function("inference_step_forward"):
        logits = model(input_ids)

    return logits

# Quick baseline timing.
start = time.perf_counter()
loss_value = train_step(model, optimizer)
if device.type == "cuda":
    torch.cuda.synchronize()
train_ms = (time.perf_counter() - start) * 1000.0

start = time.perf_counter()
infer_logits = inference_step(model)
if device.type == "cuda":
    torch.cuda.synchronize()
infer_ms = (time.perf_counter() - start) * 1000.0

print(f"Training step loss: {loss_value:.4f}")
print(f"Training step time: {train_ms:.2f} ms")
print(f"Inference step time: {infer_ms:.2f} ms")
print(f"Inference logits shape: {tuple(infer_logits.shape)}")


## 3. PyTorch profiler: the first profiler to reach for

PyTorch profiler is often the best first tool because it quickly answers:
- which operators dominate time?
- how much CPU time vs CUDA time do they consume?
- are there suspiciously many small ops?
- where is backward expensive?

If you are still asking *what part of my model is slow?*, PyTorch profiler is often the right first step.

In [ ]:

# ============================================================
# PyTorch profiler basic pass
# ============================================================
# PyTorch profiler is often the first tool to reach for because
# it tells you:
# - which operators are expensive
# - whether time is on CPU or CUDA
# - which framework regions dominate

activities = [ProfilerActivity.CPU]
if torch.cuda.is_available():
    activities.append(ProfilerActivity.CUDA)

# Use inference here so the basic example stays fast.
with profile(
    activities=activities,
    record_shapes=True,
    profile_memory=False,
    with_stack=False,
    with_modules=False,
) as prof:
    _ = inference_step(model)

sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
prof_table = prof.key_averages().table(sort_by=sort_key, row_limit=12)
print(prof_table)


### What to look for in the table

The profiler table lets you rank hotspots.

Common patterns:
- large matmuls or SDPA near the top in Transformer workloads
- many tiny ops suggesting fragmentation or Python overhead
- backward ops taking more time than expected
- CPU-heavy sections that may starve the GPU

## 4. Scheduled profiling and trace export

Profiling every step all the time can be too expensive.

That is why scheduled profiling is useful:
- **wait**: skip some early steps
- **warmup**: let the profiler stabilize
- **active**: record the interesting window

The profiler can then hand traces to a trace handler, such as TensorBoard trace export.

In [ ]:

# ============================================================
# Scheduled profiling and TensorBoard trace export
# ============================================================
# Scheduled profiling is useful when:
# - you do not want to profile every step
# - you want a controlled trace window
# - you want exportable traces

trace_dir = Path(tempfile.mkdtemp(prefix="torch_profiler_trace_"))

def trace_handler(prof: torch.profiler.profile) -> None:
    handler = torch.profiler.tensorboard_trace_handler(str(trace_dir))
    handler(prof)

scheduled_model = TinyCausalLM(vocab_size=VOCAB_SIZE, d_model=32, num_heads=4, ff_dim=64, num_layers=1, max_len=64).to(device)

with profile(
    activities=activities,
    schedule=torch.profiler.schedule(wait=1, warmup=1, active=1, repeat=1),
    on_trace_ready=trace_handler,
    record_shapes=False,
    profile_memory=False,
    with_stack=False,
) as p:
    for step_idx in range(3):
        _ = inference_step(scheduled_model)
        p.step()

trace_files = sorted([str(path.name) for path in trace_dir.glob("**/*") if path.is_file()])

print(f"Trace directory: {trace_dir}")
print("Generated trace files:")
for name in trace_files[:10]:
    print("  ", name)


## 5. Chrome trace export

A trace timeline can answer questions that a ranked operator table cannot.

For example:
- are CPU-side preparations delaying launches?
- are copies overlapping with compute?
- are kernels serialized?
- do custom regions line up with intuition?

This is where trace-style thinking starts to overlap with what engineers often use Nsight Systems for.

In [ ]:

# ============================================================
# Chrome trace export
# ============================================================
# Chrome trace is useful because it gives a timeline-style view.
# That helps bridge the mental model between framework traces and
# system-level tools such as Nsight Systems.

chrome_trace_path = trace_dir / "single_step_trace.json"

with profile(
    activities=activities,
    record_shapes=False,
    profile_memory=False,
    with_stack=False,
) as chrome_prof:
    _ = inference_step(model)

chrome_prof.export_chrome_trace(str(chrome_trace_path))

print(f"Chrome trace exists: {chrome_trace_path.exists()}")
print(f"Chrome trace path: {chrome_trace_path}")


## 6. Turning profiler output into a simple ranking view

In interviews, it is often useful to show that you can move from raw output to a prioritized action list.

In [ ]:

# ============================================================
# Turn profiler output into a ranking view
# ============================================================
# A good infra engineer does not stop at raw profiler output.
# They turn the data into a prioritized view of likely bottlenecks.

events = []
for evt in prof.key_averages():
    if torch.cuda.is_available():
        self_time_us = evt.self_cuda_time_total
    else:
        self_time_us = evt.self_cpu_time_total

    events.append(
        {
            "name": evt.key,
            "self_time_us": float(self_time_us),
            "count": int(evt.count),
        }
    )

events = sorted(events, key=lambda row: row["self_time_us"], reverse=True)

print("Top events by self time:")
for row in events[:10]:
    print(f"{row['name'][:35]:35s} time_us={row['self_time_us']:10.1f} count={row['count']:4d}")

top_k = events[:8]
labels = [row["name"][:24] for row in top_k]
values = [row["self_time_us"] for row in top_k]

plt.figure(figsize=(8, 4))
plt.barh(labels[::-1], values[::-1])
plt.xlabel("Self time (microseconds)")
plt.title("Top profiler events")
plt.tight_layout()
plt.show()


## 7. Nsight Systems intuition

**Nsight Systems** is the tool to reach for when you need a **timeline of the whole program**.

Typical reasons:
- the GPU is not busy enough
- kernels are separated by large launch gaps
- H2D copies are badly placed
- dataloader or CPU preprocessing is starving the GPU
- you want to see CPU threads, CUDA API calls, GPU kernels, and overlap in one view

In [ ]:

# ============================================================
# Nsight Systems and Nsight Compute helpers
# ============================================================
# These helpers are intentionally simple.
# The goal is to make the profiling-tool distinction memorable.

def build_nsys_command(script_name: str = "train.py") -> str:
    # Nsight Systems is a timeline-oriented system profiler.
    return "nsys profile --trace=cuda,nvtx,osrt -o profile_report python " + script_name

def build_ncu_command(script_name: str = "train.py") -> str:
    # Nsight Compute is a per-kernel profiler.
    return "ncu --set full --target-processes all python " + script_name

print("Example Nsight Systems command:")
print(" ", build_nsys_command())

print("\nExample Nsight Compute command:")
print(" ", build_ncu_command())

print("\nTool availability in this runtime:")
print("  nsys installed:", shutil.which("nsys") is not None)
print("  ncu  installed:", shutil.which("ncu") is not None)


### Reading a timeline like an infra engineer

A good timeline-reading habit is:
1. find long idle regions
2. ask whether the gap is CPU-side, transfer-side, or synchronization-side
3. look for overlap between copies and compute
4. look for launch granularity problems
5. only then decide if you need deeper kernel metrics

In [ ]:

# ============================================================
# Synthetic timeline intuition plot
# ============================================================
# Nsight Systems is all about understanding the whole-program
# timeline. This synthetic figure helps make that idea concrete.

timeline_events = [
    ("CPU input work", 0, 8, "CPU"),
    ("H2D copy", 8, 11, "Transfer"),
    ("Forward kernels", 11, 24, "GPU"),
    ("Backward kernels", 24, 39, "GPU"),
    ("Optimizer", 39, 44, "CPU"),
    ("Idle gap", 44, 54, "Idle"),
    ("CPU input work", 54, 62, "CPU"),
    ("H2D copy", 62, 65, "Transfer"),
    ("Forward kernels", 65, 77, "GPU"),
]

color_map = {
    "CPU": "#4C78A8",
    "Transfer": "#54A24B",
    "GPU": "#F58518",
    "Idle": "#B0B0B0",
}

plt.figure(figsize=(10, 2.8))
for label, start, end, kind in timeline_events:
    plt.barh(0, end - start, left=start, height=0.5, color=color_map[kind], edgecolor="black")
    plt.text(start + 0.4, 0, label, va="center", fontsize=8)

plt.yticks([])
plt.xlabel("Time (arbitrary ms)")
plt.title("Synthetic timeline intuition: what Nsight Systems helps you see")
plt.tight_layout()
plt.show()


## 8. Nsight Compute intuition

**Nsight Compute** is a **kernel profiler**.

You usually use it after you already know:
- which kernel or kernel family matters
- that the issue is likely inside the kernel itself
- that a system-wide timeline alone is not enough

### Common tool-selection pattern

A very common profiling sequence is:
1. **PyTorch profiler** to find the high-level hotspot
2. **Nsight Systems** to understand whole-program timing and overlap
3. **Nsight Compute** only for the hottest important kernel(s)

In [ ]:

# ============================================================
# Profiling decision table
# ============================================================
# In interviews, choosing the right tool is often more important
# than memorizing every profiler flag.

tool_table = [
    {
        "question": "Which PyTorch ops dominate runtime?",
        "best_first_tool": "PyTorch profiler",
        "why": "Fast operator-level hotspot ranking inside the framework",
    },
    {
        "question": "Why is the GPU idle between steps?",
        "best_first_tool": "Nsight Systems",
        "why": "Timeline view across CPU, CUDA APIs, kernels, copies, and gaps",
    },
    {
        "question": "Why is one kernel slow?",
        "best_first_tool": "Nsight Compute",
        "why": "Per-kernel metrics and deeper kernel analysis",
    },
    {
        "question": "Is the bottleneck dataloading / host-side launch overhead?",
        "best_first_tool": "Nsight Systems",
        "why": "Whole-program timeline and overlap visibility",
    },
]

print("Profiling decision table")
for row in tool_table:
    print(f"- Question: {row['question']}")
    print(f"  Best first tool: {row['best_first_tool']}")
    print(f"  Why: {row['why']}")
    print()


## 9. Practical interview framing

If an interviewer says *Training got slower after a refactor. What would you do?*

A strong answer is often:
1. confirm the regression with a simple wall-clock baseline
2. use PyTorch profiler to find changed hotspots
3. if GPU idle time or launch gaps are suspected, capture a timeline with Nsight Systems
4. if one kernel dominates and still looks suspicious, inspect it with Nsight Compute

## 10. Interview-style exercises

### Exercise 1 — Which tool first?

You see that the training step is slower than expected, but you do not yet know whether the problem is:
- one expensive operator,
- too many small operators,
- or CPU-side overhead.

Which tool is the best **first** step?

**Hint:** Ask yourself which tool gives the fastest high-level operator ranking.

### Exercise 1 — Answer

A good first step is usually **PyTorch profiler**.

Why:
- it quickly shows operator-level hotspots
- it separates CPU and CUDA time
- it is easy to run from within the framework code

### Exercise 2 — You suspect the GPU is idle between kernels. Which tool now?

**Hint:** You want a timeline across CPU work, CUDA APIs, and GPU execution.

### Exercise 2 — Answer

Use **Nsight Systems**.

Why:
- it is built for system-level timeline analysis
- it helps you see CPU-side gaps, copies, launches, synchronization, and overlap

### Exercise 3 — You already know one kernel is hot. Which tool now?

**Hint:** Now you want detailed kernel metrics, not just a timeline.

### Exercise 3 — Answer

Use **Nsight Compute**.

Why:
- it is the per-kernel profiler
- it provides low-level metrics and deeper analysis for that specific kernel

### Exercise 4 — Why not start with Nsight Compute every time?

**Hint:** Think about cost, scope, and whether you already know which kernel matters.

### Exercise 4 — Answer

Because Nsight Compute is most useful when you already know:
- which kernel matters
- that the issue is probably inside the kernel

If you start there too early, you may spend time collecting deep metrics for a kernel that is not actually the main bottleneck.

### Exercise 5 — Small coding challenge

Modify the tiny model so it has **4 decoder blocks instead of 1**, then rerun the profiler.

What changes would you expect in the hotspot table?

**Hint:** Think about repeated attention and MLP blocks.

### Exercise 5 — Answer

With more decoder blocks, you would usually expect:
- attention-related ops to appear more often or take more total time
- MLP/GELU / linear ops to take more total time
- overall step time to increase

### Exercise 6 — Interview reasoning challenge

Suppose PyTorch profiler says one attention-related operator dominates runtime, but Nsight Systems also shows long idle gaps between training steps.

What does that suggest?

**Hint:** There may be more than one bottleneck layer.

### Exercise 6 — Answer

It suggests there may be **both**:
- an expensive compute hotspot inside the model
- and a systems-level efficiency problem such as CPU launch gaps, data delays, or synchronization overhead

## 11. Final takeaways

For infra interviews, try to be able to say these clearly:
1. PyTorch profiler is usually the best first tool for framework/operator hotspots
2. Nsight Systems is for timeline-level CPU/GPU concurrency and idle-gap analysis
3. Nsight Compute is for deep per-kernel analysis
4. Profiling should usually proceed from **broad** to **narrow**

## 12. Suggested next steps

After this notebook, strong follow-ups are:
- profile training vs inference separately
- increase sequence length and see how hotspots move
- export traces to TensorBoard and inspect them visually
- add custom NVTX / `record_function` regions around dataloading, forward, backward, and optimizer
- if you have a local NVIDIA setup, try one real `nsys` capture and one targeted `ncu` run

## References to read next

For the most authoritative follow-up reading, use the official docs for:
- `torch.profiler`
- the PyTorch profiler TensorBoard tutorial
- NVIDIA Nsight Systems User Guide
- NVIDIA Nsight Compute docs